# 01 — Exploratory Data Analysis: Credit Card Fraud Detection

**Projekt:** Sustav za personaliziranu detekciju anomalija u sekvencijalnim podacima  
**Dataset:** [Credit Card Fraud Detection — Kaggle (ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Cilj ovog notebooka:**
- Razumjeti strukturu i distribuciju podataka
- Identificirati klasu neravnoteže (class imbalance)
- Vizualizirati razlike između normalnih i lažnih transakcija
- Donijeti odluke o preprocessingu za sljedeće notebookove

**Reference:**  
Dal Pozzolo et al. (2015). *Calibrating Probability with Undersampling for Unbalanced Classification*. IEEE SSCI.

In [ ]:
# ============================================================
# 1. INSTALACIJA PAKETA
# Pokrenuti samo jednom pri prvom pokretanju u Colabu
# ============================================================
!pip install kagglehub pandas numpy matplotlib seaborn scikit-learn -q
print("✅ Paketi instalirani")

In [ ]:
# ============================================================
# 2. IMPORTS
# ============================================================
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Globalne postavke vizualizacije
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 5)

print("✅ Imports uspješni")

In [ ]:
# ============================================================
# 3. UČITAVANJE DATASETA
# kagglehub automatski skida i cachira dataset
# ============================================================
print("⏳ Preuzimanje dataseta...")
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print(f"📁 Dataset path: {path}")

csv_path = Path(path) / "creditcard.csv"
df = pd.read_csv(csv_path)

print(f"\n✅ Dataset učitan!")
print(f"   Dimenzije: {df.shape[0]:,} redaka × {df.shape[1]} stupaca")

## 3.1 Osnovna inspekcija podataka

In [ ]:
# Prikaz prvih 5 redaka
print("=== PRVIH 5 REDAKA ===")
df.head()

In [ ]:
# Tipovi podataka i broj non-null vrijednosti
print("=== INFO ===")
df.info()

In [ ]:
# Statistički sažetak numeričkih stupaca
print("=== STATISTIČKI SAŽETAK ===")
df.describe().T.style.background_gradient(cmap="Blues")

## 3.2 Provjera nedostajućih vrijednosti

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Nedostajuće vrijednosti": missing,
    "Postotak (%)": missing_pct
}).query("`Nedostajuće vrijednosti` > 0")

if missing_df.empty:
    print("✅ Nema nedostajućih vrijednosti — dataset je čist!")
else:
    print("⚠️ Pronađene nedostajuće vrijednosti:")
    display(missing_df)

## 3.3 Analiza klasa (Class Imbalance)

Ovo je **ključni uvid** za detekciju anomalija: prijevare su ekstremno rijetke.  
Takav nerazmjer klasa direktno utječe na odabir modela i metrika evaluacije.

In [ ]:
class_counts = df["Class"].value_counts()
class_pct    = df["Class"].value_counts(normalize=True) * 100

class_summary = pd.DataFrame({
    "Klasa":   ["Normalna transakcija (0)", "Prijevara (1)"],
    "Broj":    [class_counts[0], class_counts[1]],
    "Udio (%)": [class_pct[0].round(4), class_pct[1].round(4)]
})
print(class_summary.to_string(index=False))

# ── Vizualizacija ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(["Normalna (0)", "Prijevara (1)"],
            [class_counts[0], class_counts[1]],
            color=["#2196F3", "#F44336"], edgecolor="white", width=0.5)
axes[0].set_title("Distribucija klasa", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Broj transakcija")
for i, v in enumerate([class_counts[0], class_counts[1]]):
    axes[0].text(i, v + 1000, f"{v:,}", ha="center", fontsize=10)

# Pie chart
axes[1].pie(
    [class_counts[0], class_counts[1]],
    labels=["Normalna", "Prijevara"],
    colors=["#2196F3", "#F44336"],
    autopct="%1.3f%%",
    startangle=90,
    explode=(0, 0.1)
)
axes[1].set_title("Udio klasa (%)", fontsize=13, fontweight="bold")

plt.suptitle("Class Imbalance — Credit Card Fraud Dataset", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("class_distribution.png", bbox_inches="tight")
plt.show()

print(f"\n⚠️  Omjer neravnoteže: 1 prijevara na svake "
      f"{int(class_counts[0]/class_counts[1])} normalnih transakcija")

## 3.4 Pregled značajki (Features)

Dataset sadrži:
- **Time** — sekunde od prve transakcije
- **V1–V28** — PCA-transformirane značajke (originalni podaci su anonomizirani zbog privatnosti)
- **Amount** — iznos transakcije u EUR
- **Class** — 0 = normalna, 1 = prijevara (ciljna varijabla)

In [ ]:
print(f"Broj značajki (features): {df.shape[1] - 1}")
print(f"Ciljna varijabla: 'Class'")
print(f"\nPCA značajke: V1 – V28 (28 stupaca)")
print(f"Ostale značajke: Time, Amount")
print(f"\nRaspon 'Time':   {df['Time'].min():.0f} – {df['Time'].max():.0f} sekundi")
print(f"Raspon 'Amount': {df['Amount'].min():.2f} – {df['Amount'].max():.2f} EUR")
print(f"Prosječni iznos: {df['Amount'].mean():.2f} EUR")

## 3.5 Analiza iznosa transakcija (Amount)

In [ ]:
normal = df[df["Class"] == 0]["Amount"]
fraud  = df[df["Class"] == 1]["Amount"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribucija iznosa — normalne vs prijevare
axes[0].hist(normal, bins=50, alpha=0.6, color="#2196F3",
             label=f"Normalne (n={len(normal):,})", density=True)
axes[0].hist(fraud,  bins=50, alpha=0.7, color="#F44336",
             label=f"Prijevare (n={len(fraud):,})", density=True)
axes[0].set_xlabel("Iznos (EUR)")
axes[0].set_ylabel("Gustoća")
axes[0].set_title("Distribucija iznosa transakcija")
axes[0].set_xlim(0, 500)
axes[0].legend()

# Box plot usporedba
axes[1].boxplot(
    [normal.clip(upper=500), fraud.clip(upper=500)],
    labels=["Normalna", "Prijevara"],
    patch_artist=True,
    boxprops=dict(facecolor="#E3F2FD"),
    medianprops=dict(color="#F44336", linewidth=2)
)
axes[1].set_ylabel("Iznos (EUR, clipped na 500)")
axes[1].set_title("Box plot iznosa po klasi")

plt.suptitle("Analiza iznosa transakcija", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("amount_analysis.png", bbox_inches="tight")
plt.show()

print("\n=== STATISTIKE IZNOSA ===")
comp = pd.DataFrame({
    "Normalne": normal.describe(),
    "Prijevare": fraud.describe()
}).round(2)
display(comp)

## 3.6 Analiza vremenske distribucije

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Normalne transakcije kroz vrijeme
axes[0].hist(df[df["Class"] == 0]["Time"] / 3600,
             bins=96, color="#2196F3", alpha=0.7)
axes[0].set_ylabel("Broj transakcija")
axes[0].set_title("Normalne transakcije kroz vrijeme")

# Prijevare kroz vrijeme
axes[1].hist(df[df["Class"] == 1]["Time"] / 3600,
             bins=96, color="#F44336", alpha=0.7)
axes[1].set_ylabel("Broj prijevara")
axes[1].set_xlabel("Vrijeme (sati)")
axes[1].set_title("Prijevarne transakcije kroz vrijeme")

plt.suptitle("Vremenska distribucija transakcija (48h period)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("time_distribution.png", bbox_inches="tight")
plt.show()

print("💡 Uočljivo: postoje dnevni obrasci u normalnim transakcijama.")
print("   Prijevare su distribuirane ravnomjernije — nema jakog vremenskog obrasca.")

## 3.7 Korelacija značajki s ciljnom varijablom

In [ ]:
# Korelacija svakog featurea s Class varijablom
correlations = df.corr()["Class"].drop("Class").sort_values()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#F44336" if c < 0 else "#2196F3" for c in correlations]
ax.barh(correlations.index, correlations.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Pearson korelacija s 'Class'")
ax.set_title("Korelacija značajki s Class (prijevara=1)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("feature_correlation.png", bbox_inches="tight")
plt.show()

top_pos = correlations.nlargest(5)
top_neg = correlations.nsmallest(5)
print("\n🔴 Top 5 negativno korelirani (indikatori prijevare):")
print(top_neg.round(4).to_string())
print("\n🔵 Top 5 pozitivno korelirani:")
print(top_pos.round(4).to_string())

## 3.8 Distribucija top značajki po klasi

Vizualiziramo razlike u distribuciji između normalnih i lažnih transakcija  
za najkoreliranijih 6 značajki.

In [ ]:
# Odabir top 6 najkoreliranijih značajki (po apsolutnoj vrijednosti)
top_features = correlations.abs().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    ax = axes[i]
    ax.hist(df[df["Class"] == 0][feature], bins=50,
            alpha=0.6, color="#2196F3", label="Normalna", density=True)
    ax.hist(df[df["Class"] == 1][feature], bins=50,
            alpha=0.7, color="#F44336", label="Prijevara", density=True)
    ax.set_title(f"{feature}  (corr={correlations[feature]:.3f})")
    ax.set_xlabel("Vrijednost")
    ax.set_ylabel("Gustoća")
    ax.legend(fontsize=8)

plt.suptitle("Distribucija top 6 značajki: Normalna vs Prijevara",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("top_features_distribution.png", bbox_inches="tight")
plt.show()

print(f"\n✅ Top 6 značajki: {top_features}")
print("   Ove značajke imat će najveći utjecaj na detekciju anomalija.")

## 3.9 Odluke o preprocessingu

Na temelju EDA, definiramo preprocessing pipeline za sljedeće notebookove.

In [ ]:
# ============================================================
# PREPROCESSING ODLUKE — temelj za notebook 02 i 03
# ============================================================

# 1. Feature selekcija — koristimo sve V1-V28 + Amount, izbacujemo Time
#    Razlog: Time nije dobar prediktor (prijevare se događaju u svako doba)
features_to_use = [f"V{i}" for i in range(1, 29)] + ["Amount"]
print(f"✅ Features za modeliranje: {len(features_to_use)} značajki")
print(f"   {features_to_use}")

# 2. Skaliranje — Amount ima potpuno drugačiji raspon od V1-V28
#    StandardScaler jer V1-V28 su već centirirani (PCA), samo Amount treba skaliranje
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[["Amount"]] = scaler.fit_transform(df[["Amount"]])
print(f"\n✅ Amount skaliran — mean: {df['Amount'].mean():.2f} → 0.00")

# 3. Podjela na normalne i anomalije
normal_data = df_scaled[df_scaled["Class"] == 0][features_to_use]
fraud_data  = df_scaled[df_scaled["Class"] == 1][features_to_use]
print(f"\n✅ Normalne transakcije: {len(normal_data):,}")
print(f"   Prijevare:            {len(fraud_data):,}")

# 4. Train/validation/test split — PO VREMENU, ne nasumično!
#    70% train / 15% validation / 15% test
#    Razlog: nasumična podjela narušava vremensku strukturu
n = len(df_scaled)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

df_train = df_scaled.iloc[:train_end]
df_val   = df_scaled.iloc[train_end:val_end]
df_test  = df_scaled.iloc[val_end:]

print(f"\n✅ Train/Val/Test split (po vremenu):")
print(f"   Train: {len(df_train):,} ({len(df_train)/n*100:.0f}%)")
print(f"   Val:   {len(df_val):,}   ({len(df_val)/n*100:.0f}%)")
print(f"   Test:  {len(df_test):,}  ({len(df_test)/n*100:.0f}%)")

# 5. Provjera distribucije klasa po splitovima
for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    fraud_count = split["Class"].sum()
    print(f"   {name} — prijevare: {fraud_count} ({fraud_count/len(split)*100:.3f}%)")

## 3.10 Spremanje preprocessiranih podataka

In [ ]:
import os, joblib

# Kreiraj output direktorij
os.makedirs("../data/credit_card/processed", exist_ok=True)

# Spremi splitove
df_train.to_csv("../data/credit_card/processed/train.csv", index=False)
df_val.to_csv("../data/credit_card/processed/val.csv",   index=False)
df_test.to_csv("../data/credit_card/processed/test.csv", index=False)

# Spremi scaler za korištenje u backendu
joblib.dump(scaler, "../data/credit_card/processed/scaler.pkl")

# Spremi listu featurea
import json
with open("../data/credit_card/processed/features.json", "w") as f:
    json.dump({"features": features_to_use}, f, indent=2)

print("✅ Svi artefakti spremljeni:")
print("   ../data/credit_card/processed/train.csv")
print("   ../data/credit_card/processed/val.csv")
print("   ../data/credit_card/processed/test.csv")
print("   ../data/credit_card/processed/scaler.pkl")
print("   ../data/credit_card/processed/features.json")

## 4. Zaključak EDA

### Ključni uvidi

| Uvid | Detalj |
|---|---|
| **Ekstremna neravnoteža klasa** | 0.172% prijevara — model mora biti robustan na ovo |
| **Anonomizirani podaci** | V1–V28 su PCA komponente, ne možemo interpretirati direktno |
| **Amount je informavan** | Prijevare imaju nešto drugačiju distribuciju iznosa |
| **Time nije ključan prediktor** | Prijevare se događaju ravnomjerno kroz dan |
| **Top prediktori** | V17, V14, V12, V10 imaju najveću negativnu korelaciju s prijevarama |

### Odluke za sljedeće notebookove

1. **Features:** V1–V28 + Amount (bez Time)
2. **Skaliranje:** StandardScaler samo na Amount
3. **Split:** Po vremenu (70/15/15), ne nasumično
4. **Metrika evaluacije:** F1 score i ROC-AUC (ne accuracy — zbog class imbalance!)
5. **Isolation Forest** (notebook 02) — baseline model
6. **LSTM Autoenkoder** (notebook 03) — glavni model, trenira se samo na normalnim podacima

### Sljedeći korak
➡️ Otvori `02_baseline_isolation_forest.ipynb`